In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
TEACHER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_MODEL
)

In [3]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
prompt = "What is python?"
inputs = tokenizer(
    prompt, 
    return_tensors = 'pt'
)

with torch.no_grad():

    outputs = teacher(
        **inputs
    )

In [13]:
# outputs = teacher(
#     **inputs
# )

print(inputs)

{'input_ids': tensor([[ 3838,   374, 10135,    30]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [9]:
print(outputs)

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 6.6250,  7.7812,  6.7812,  ..., -0.8359, -0.8359, -0.8359],
         [ 7.3125,  8.2500,  6.0312,  ...,  1.1250,  1.1250,  1.1250],
         [ 9.8750,  8.8750,  9.0000,  ...,  1.5781,  1.5781,  1.5781],
         [ 8.5000,  6.7500, 11.4375,  ..., -3.0938, -3.0938, -3.0938]]],
       dtype=torch.bfloat16), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)


In [10]:
logits = outputs.logits
print(logits.shape)

# 1              → batch size
# 4              → number of input tokens
# 151936         → vocabulary size

torch.Size([1, 4, 151936])


In [14]:
Student_model = "Qwen/Qwen2.5-0.5B-Instruct"

student_tokenizer = AutoTokenizer.from_pretrained(
    Student_model
)

student = AutoModelForCausalLM.from_pretrained(
    Student_model
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [18]:
teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False 


In [19]:
student.train()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [2]:
import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)


TEACHER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
STUDENT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

TEMPERATURE = 2.0


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# --------------------------------------------------
# Tokenizer
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL
)


# --------------------------------------------------
# Teacher
# --------------------------------------------------

teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL
)

teacher.to(device)

teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False


# --------------------------------------------------
# Student
# --------------------------------------------------

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL
)

student.to(device)

student.train()


# --------------------------------------------------
# Example input
# --------------------------------------------------

prompt = "Explain what a Python decorator is."


inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


# --------------------------------------------------
# Teacher forward pass
# --------------------------------------------------

with torch.no_grad():

    teacher_outputs = teacher(
        **inputs
    )


teacher_logits = teacher_outputs.logits


# --------------------------------------------------
# Student forward pass
# --------------------------------------------------

student_outputs = student(
    **inputs
)

student_logits = student_outputs.logits


# --------------------------------------------------
# Convert logits to distributions
# --------------------------------------------------

teacher_probs = F.softmax(
    teacher_logits / TEMPERATURE,
    dim=-1
)


student_log_probs = F.log_softmax(
    student_logits / TEMPERATURE,
    dim=-1
)


# --------------------------------------------------
# KL divergence
# --------------------------------------------------

loss = F.kl_div(
    student_log_probs,
    teacher_probs,
    reduction="batchmean"
)


distillation_loss = (
    TEMPERATURE ** 2
) * loss


print(
    "Distillation loss:",
    distillation_loss.item()
)

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mahesh.shriram\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\s

Distillation loss: 18.0


In [6]:
optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=5e-5
)


In [7]:
optimizer.zero_grad()

student_outputs = student(
    **inputs
)

student_logits = student_outputs.logits

student_log_probs = F.log_softmax(
    student_logits / TEMPERATURE,
    dim=-1
)

loss = F.kl_div(
    student_log_probs,
    teacher_probs,
    reduction="batchmean"
)

distillation_loss = (
    TEMPERATURE ** 2
) * loss

distillation_loss.backward()

optimizer.step()